<a href="https://colab.research.google.com/github/IrumShehryar/ML-NLP-Coursework/blob/main/nlp/04-neural-network/markov_model-for-text-generation/Project02_FirstOrder_Markov_Text_Generation_Small_Corpus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries

In [1]:
import numpy as np
import string # for creating string

In [ ]:
# This code is inspired from the following References
# https://www.kaggle.com/wildflowerhd/text-generation-using-markov-chain-algorithm
# https://harjot-dadhwal.medium.com/text-generation-using-markov-chain-algorithm-ec99ee8561d1

# Get the data and tokenize it

In [2]:
text = " The quick brown fox jumps over the lazy dog and the quick white cat jumps over the lazy rabbit"

In [3]:
tokens = text.lower().split()

In [4]:
tokens

['the',
 'quick',
 'brown',
 'fox',
 'jumps',
 'over',
 'the',
 'lazy',
 'dog',
 'and',
 'the',
 'quick',
 'white',
 'cat',
 'jumps',
 'over',
 'the',
 'lazy',
 'rabbit']

In [5]:
T = len(tokens)

In [6]:
T

19

# Initialize the distribution dictionaries

In [7]:
pi = {} # Initial distribution representing the start of the sentencee
A = {} # First order transitions

# Create the function that will fill the above dictionaries

In [8]:
def fill_dict(d, k, v): # Three inputs are dictionary, keys and values. Keys represent some starting words.  First we collect the words
                         # from text and then assign prob to the words.
  if k not in d:
    d[k] = []
  d[k].append(v)        # We are collecting the word here

# Iterate over the text to fill the dictionaries

In [9]:
for i in range(T):
    t = tokens[i] # Grab the ith token
    if i == 0: # This is the first word in sentence so we would like to update our first word in distribution.
               # we can do it by incrementing the value stored for t by one. t is token and it is a key in the dictionary. we are adding 1 to its value.
               # If t does not exist in the dictionary of get then it becomes zero
      pi[t] = pi.get(t, 0.) + 1   # get will ensure that if t is not present in dictionary, it automatically get the count of 0.
                                                      # The first step is only to gather the counts. After getting through the whole dataset we can
                                                      # normalize this distribution.
    else:
      t_1 = tokens[i-1]   # If i is not 0, we will grab the previous word in sentence
    if(i != 0):
      fill_dict(A, t_1, t) # Here t_1 is the previous word and t is the current word

    if i == T-1 :      # check if we are at the end of the sentence. if i = T-1, we are at the end of the line.
        fill_dict(A,t,'<end>') # Here we are creating a fake token "<end>". This "<end>" will ensure that our line should end during data generation
                                         #if we sample the end token. This is required otherwise our line will go forever.

# Observe the distributions

In [10]:
pi # This will give the count of the first word

{'the': 1.0}

In [11]:
A

{'the': ['quick', 'lazy', 'quick', 'lazy'],
 'quick': ['brown', 'white'],
 'brown': ['fox'],
 'fox': ['jumps'],
 'jumps': ['over', 'over'],
 'over': ['the', 'the'],
 'lazy': ['dog', 'rabbit'],
 'dog': ['and'],
 'and': ['the'],
 'white': ['cat'],
 'cat': ['jumps'],
 'rabbit': ['<end>']}

# Create a function to convert list of possible words to dictionary of probabilities

In [12]:
def list_to_probdict(tokens): # This function does two things. First it creates the dictionary of count and then normalize
                              # the counts to convert each count into probability. The input to the function is tokens which is
                              # list of the token
  d = {}
  T = len(tokens)              # The total number of samples which is length of tokens
  for t in tokens:             # loop through each token
    d[t] = d.get(t, 0.) + 1   # increment the value by 1 each time we encounter the token. After this for loop we have the dictionary
                              # of counts where key is the token and value is the corressponding count
  for t, c in d.items():
    d[t] = c / T
  return d

# Apply the function to first order transitions

In [13]:
# Now we have a first order dictionary which stores second word of each sentence. we have previously stored in this
# dictionery is the list of possible next tokens. we replace the list of tokens by dictionary of probabilities
# by applying the function list_to_prob_dict
for t_1, token in A.items():
  # replace list with dictionary of probabilities
  A[t_1] = list_to_probdict(token)

In [15]:
t_1

'rabbit'

In [16]:
token

['<end>']

In [14]:
A

{'the': {'quick': 0.5, 'lazy': 0.5},
 'quick': {'brown': 0.5, 'white': 0.5},
 'brown': {'fox': 1.0},
 'fox': {'jumps': 1.0},
 'jumps': {'over': 1.0},
 'over': {'the': 1.0},
 'lazy': {'dog': 0.5, 'rabbit': 0.5},
 'dog': {'and': 1.0},
 'and': {'the': 1.0},
 'white': {'cat': 1.0},
 'cat': {'jumps': 1.0},
 'rabbit': {'<end>': 1.0}}

# Create a function for sampling the words

In [17]:
def sample_word(d): # Here input "d" is dictionary of probability. key is the possible word and value is the corresponding probability
  p0 = np.random.random() # draw a sample from uniform distribution. This is number between 0 and 1.
  cumulative = 0
  for t, p in d.items(): # t is the word and p is the corresponding probability
    cumulative += p      # incrementing commulative sum by p.
    if p0 < cumulative:  # check if p0 is less than the commulative sum. if it is then we should return a current token t
      return t

# Create a function to generate the text

In [18]:
def generateText():
  for i in range(3): # generate 3 lines
    sentence = [] # it will store the tokens we generate

    # sample initial word

    w0 = sample_word(pi)
    sentence.append(w0)

    # First-order transitions until <end>

    while True:
      w1 = sample_word(A[(w0)]) # we want w1 to depend on the first or previous word we generated so we index first order using w0.
                                # This will give us back the probaility dictionary. we pass into our sample_word function once again.
                                # This will give us the secind word in the sentence which iS w1. we will continue untill we reach the
                                # end token.
      if w1 == '<end>':
        break
      sentence.append(w1)
      w0 = w1 # update the previous word
    print(' '.join(sentence))

# Generate three lines of text

In [29]:
generateText()

the quick brown fox jumps over the lazy dog and the quick white cat jumps over the lazy dog and the lazy dog and the lazy dog and the lazy rabbit
the quick brown fox jumps over the lazy rabbit
the lazy rabbit
